In [396]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd 
import numpy as np 

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit

In [397]:
df = pd.read_csv('datasets/weatherAUS.csv')
df.head()

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


In [398]:
df = df.loc[:, ['Location', 'MaxTemp', 'Date']]
df

,Location,MaxTemp,Date
0,Albury,22.9,2008-12-01
1,Albury,25.1,2008-12-02
2,Albury,25.7,2008-12-03
3,Albury,28.0,2008-12-04
4,Albury,32.3,2008-12-05
...,...,...,...
145455,Uluru,23.4,2017-06-21
145456,Uluru,25.3,2017-06-22
145457,Uluru,26.9,2017-06-23
145458,Uluru,27.0,2017-06-24


In [399]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 145460 entries, 0 to 145459
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   Location  145460 non-null  str    
 1   MaxTemp   144199 non-null  float64
 2   Date      145460 non-null  str    
dtypes: float64(1), str(2)
memory usage: 3.3 MB


In [400]:
df.dropna(inplace=True)

In [401]:
df.sort_values('Date', inplace=True)

In [402]:
states = df['Location'].unique()
df_states = []
states

<StringArray>
[        'Canberra',           'Sydney',         'Adelaide',
            'Perth',           'Hobart',           'Darwin',
        'Melbourne',         'Brisbane',     'AliceSprings',
        'GoldCoast',           'Albany',       'Townsville',
         'Ballarat',          'Bendigo',      'MountGinini',
           'Albury',          'Penrith',     'MountGambier',
      'Tuggeranong',           'Cairns',       'Launceston',
        'Newcastle',       'Wollongong',            'Moree',
    'NorfolkIsland',         'Watsonia',      'Williamtown',
       'SalmonGums',       'PearceRAAF',         'Richmond',
     'CoffsHarbour',        'Nuriootpa',    'BadgerysCreek',
     'PerthAirport',         'Portland',       'WaggaWagga',
          'Woomera', 'MelbourneAirport',            'Cobar',
    'SydneyAirport',        'NorahHead',          'Mildura',
             'Sale',          'Walpole',      'Witchcliffe',
         'Dartmoor',            'Uluru',             'Nhil',
        'K

In [403]:
for i in range(len(states)):
    df_states.append(pd.DataFrame(df[ df['Location'] == state ]))
    
canebras = pd.DataFrame(df_states[0])

In [410]:
def week_maker(df):
    dic = {}
    for i in range(7, 0, -1):
        dic[f"d{8 - i}"] = df["MaxTemp"].shift(i)
    return dic

In [405]:
weeks = []
for dfs in df_states:
    if (ext:= (len(dfs) % 7)) != 0:
        dfs = dfs[: (-1 * ext)]
    weeks.append(pd.DataFrame(week_maker(dfs), ))
    

In [406]:
weeks[0].head(10)

,d1,d2,d3,d4,d5,d6,d7
45587,NaN,NaN,NaN,NaN,NaN,NaN,NaN
45588,NaN,NaN,NaN,NaN,NaN,NaN,24.3
45589,NaN,NaN,NaN,NaN,NaN,24.3,26.9
45590,NaN,NaN,NaN,NaN,24.3,26.9,23.4
45591,NaN,NaN,NaN,24.3,26.9,23.4,15.5
45592,NaN,NaN,24.3,26.9,23.4,15.5,16.1
45593,NaN,24.3,26.9,23.4,15.5,16.1,16.9
45594,24.3,26.9,23.4,15.5,16.1,16.9,18.2
45595,26.9,23.4,15.5,16.1,16.9,18.2,17.0
45596,23.4,15.5,16.1,16.9,18.2,17.0,19.5


In [409]:
for week in weeks:
    week.reset_index(drop=True, inplace=True)
    week.drop(range(7), inplace=True)    
    week.reset_index(drop=True, inplace=True)


In [411]:
weeks[0]

,d1,d2,d3,d4,d5,d6,d7
0,17.0,19.5,22.8,25.2,27.3,27.9,30.9
1,19.5,22.8,25.2,27.3,27.9,30.9,31.2
2,22.8,25.2,27.3,27.9,30.9,31.2,32.1
3,25.2,27.3,27.9,30.9,31.2,32.1,31.2
4,27.3,27.9,30.9,31.2,32.1,31.2,30.0
...,...,...,...,...,...,...,...
3411,16.0,15.2,15.7,15.3,9.9,11.5,13.0
3412,15.2,15.7,15.3,9.9,11.5,13.0,15.0
3413,15.7,15.3,9.9,11.5,13.0,15.0,15.7
3414,15.3,9.9,11.5,13.0,15.0,15.7,13.3


In [ ]:
range(7)

range(0, 7)